# 01b · Pipeline check with OpenVLA-7B (Step A3)

**Use a fresh runtime, separate from the SmolVLA notebooks.** OpenVLA needs `transformers==4.40.1`, which conflicts with LeRobot (plan Section 2.3).

- GPU: A100 (bf16, ~16 GB). On a T4 add `--load-in-4bit` to the run cell.
- Checkpoints: `openvla-7b-finetuned-libero-spatial` (T1, T2) and `-goal` (T3), ~15 GB each, downloaded once to `MyDrive/groundedvla/models/`.
- Expected: most episodes succeed. If they do, the simulator, initial states, image flip and success detection are correct, independently of LeRobot.

In [ ]:
# ---- edit once ----
REPO_URL = "https://github.com/kaustubhadhe1206/GroundedVLA.git"   # public repo, no token needed
BRANCH = "main"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
REPO = "/content/GroundedVLA"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO)
!git log -1 --format="commit %h  %s"

In [ ]:
!apt-get -qq install -y libegl1 libgl1 > /dev/null
!pip install -q -r requirements-openvla.txt
!pip install -q -e . --no-deps

In [ ]:
from pathlib import Path
from groundedvla.utils.colab import setup_session, stage_to_local
paths = setup_session()          # env vars (EGL, caches on local disk), Drive folders, LIBERO config

In [ ]:
for folder in ["openvla-7b-finetuned-libero-spatial", "openvla-7b-finetuned-libero-goal"]:
    if (Path(paths["models"]) / folder / "config.json").exists():
        stage_to_local(f"models/{folder}")

In [ ]:
EPISODES = 10
EXTRA = ""          # "--load-in-4bit" on a T4
for task in ["T1", "T2", "T3"]:
    !python scripts/run_sim.py openvla_7b --task {task} --episodes {EPISODES} --viewer none --save-video {EXTRA}

In [ ]:
from pathlib import Path
from IPython.display import Video, display
for v in sorted(Path(paths["outputs"]).glob("run_sim/openvla_7b/videos/*.mp4"))[:3]:
    print(v.name); display(Video(str(v), embed=True, width=512))

In [ ]:
import json
import pandas as pd
rows = []
for f in Path(paths["outputs"], "run_sim").glob("*/*.jsonl"):
    rows += [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
df = pd.DataFrame(rows)
df.groupby(["variant", "task"]).agg(episodes=("success", "size"), success=("success", "mean"),
                                    steps=("steps", "mean"), policy_call_ms=("latency_ms_median", "median"))